# Stage 14: Denominator Alignment and Energy Localization

Stage 13 established that the enormous local gain is caused by directional anisotropy of the $D^{-1}$ operator. The orthogonal SOAP back-projection preserves the norm, meaning the gain simplifies to:

$$ G_m \approx \frac{\|D^{-1}u\|}{\|D^{-1}q_m\|} $$

This notebook tests whether the MoQ perturbation $u$ systematically aligns more strongly with small-$v$ coordinates than the baseline moment $q_m$. We measure:
1. Coordinate correlation: $\operatorname{corr}(|u|, D^{-1})$ vs $\operatorname{corr}(|q_m|, D^{-1})$
2. Cosine similarity: $\cos(D^{-1}u, D^{-1}q_m)$
3. Energy localization ($F_q$): The fraction of transformed energy $\|D^{-1}x\|^2$ that falls into the coordinates corresponding to the smallest 5% and 10% of $v$.

If $F_q(u) \gg F_q(q_m)$, we have definitively proven that MoQ extrapolation pushes momentum perturbation directly into the most sensitive, low-confidence coordinates of the optimizer state.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class InstrumentedOfficialSOAPStage14(SOAP):
    def __init__(self, params, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.0, damping=1e-3):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.mu = mu
        self.damping = damping
        self.layer_diagnostics = []

    def compute_energy_localization(self, raw_g, moq_g, state, group, p):
        if "exp_avg" not in state or "Q" not in state or state["Q"] is None:
            return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
            
        # 1. Project both
        proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        proj_moq = self.project(moq_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        # 2. Compute moments for both
        b1, b2 = group["betas"]
        m_old, v_old = state["exp_avg"], state["exp_avg_sq"]
        
        m_raw = m_old * b1 + proj_raw * (1.0 - b1)
        m_moq = m_old * b1 + proj_moq * (1.0 - b1)
        
        v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
        denom_raw = v_raw.sqrt() + group["eps"]
        
        # Compute vectors
        diff_m = m_moq - m_raw
        diff_m_norm = float(torch.linalg.vector_norm(diff_m))
        u = diff_m / diff_m_norm if diff_m_norm > 1e-12 else torch.zeros_like(diff_m)
        
        m_raw_norm = float(torch.linalg.vector_norm(m_raw))
        q_m = m_raw / m_raw_norm if m_raw_norm > 1e-12 else torch.zeros_like(m_raw)
        
        D_inv = 1.0 / denom_raw
        
        z_u = D_inv * u
        z_m = D_inv * q_m
        
        norm_z_u = float(torch.linalg.vector_norm(z_u))
        norm_z_m = float(torch.linalg.vector_norm(z_m))
        
        r_t = norm_z_u / (norm_z_m + 1e-12)
        cos_z = float(torch.sum(z_u * z_m) / ((norm_z_u * norm_z_m) + 1e-12))
        
        u_abs = u.abs().flatten().to(torch.float32)
        q_m_abs = q_m.abs().flatten().to(torch.float32)
        D_inv_flat = D_inv.flatten().to(torch.float32)
        
        # Correlations
        def pearson_corr(x, y):
            if x.numel() < 2: return 0.0
            x_m = x - x.mean()
            y_m = y - y.mean()
            return float(torch.sum(x_m * y_m) / (torch.linalg.vector_norm(x_m) * torch.linalg.vector_norm(y_m) + 1e-12))
            
        corr_u_D = pearson_corr(u_abs, D_inv_flat)
        corr_qm_D = pearson_corr(q_m_abs, D_inv_flat)
        
        # Energy localization
        denom_flat = denom_raw.flatten()
        k5 = max(1, int(0.05 * denom_flat.numel()))
        k10 = max(1, int(0.10 * denom_flat.numel()))
        
        # Smallest denominator elements (smallest v -> largest D_inv)
        _, indices = torch.topk(denom_flat, k10, largest=False)
        idx5 = indices[:k5]
        idx10 = indices
        
        z_u_sq = (z_u.flatten() ** 2)
        z_m_sq = (z_m.flatten() ** 2)
        
        sum_z_u_sq = float(z_u_sq.sum()) + 1e-12
        sum_z_m_sq = float(z_m_sq.sum()) + 1e-12
        
        F5_u = float(z_u_sq[idx5].sum()) / sum_z_u_sq
        F5_qm = float(z_m_sq[idx5].sum()) / sum_z_m_sq
        
        F10_u = float(z_u_sq[idx10].sum()) / sum_z_u_sq
        F10_qm = float(z_m_sq[idx10].sum()) / sum_z_m_sq
        
        return r_t, cos_z, corr_u_D, corr_qm_D, F5_u, F5_qm, F10_u, F10_qm

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        moq_grads = []
        raw_grads = []
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                raw_g = p.grad.clone()
                raw_grads.append(raw_g)
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_moq = raw_g.clone()
                    state['current_moq_pred_error'] = 0.0
                    
                    state['r_t'] = 0.0
                    state['cos_z'] = 0.0
                    state['corr_u_D'] = 0.0
                    state['corr_qm_D'] = 0.0
                    state['F5_u'] = 0.0
                    state['F5_qm'] = 0.0
                    state['F10_u'] = 0.0
                    state['F10_qm'] = 0.0
                else:
                    g_moq = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    
                    if 'moq_pred' in state:
                        pred_error = float(torch.linalg.vector_norm(raw_g - state['moq_pred']))
                        state['current_moq_pred_error'] = pred_error / (float(torch.linalg.vector_norm(raw_g)) + 1e-12)
                    else:
                        state['current_moq_pred_error'] = 0.0
                        
                    state['moq_pred'] = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    state['prev_raw_g'].copy_(raw_g)
                    
                    if 'Q' in state and state['Q'] is not None:
                        r_t, cos_z, corr_u_D, corr_qm_D, F5_u, F5_qm, F10_u, F10_qm = self.compute_energy_localization(raw_g, g_moq, state, group, p)
                        state['r_t'] = r_t
                        state['cos_z'] = cos_z
                        state['corr_u_D'] = corr_u_D
                        state['corr_qm_D'] = corr_qm_D
                        state['F5_u'] = F5_u
                        state['F5_qm'] = F5_qm
                        state['F10_u'] = F10_u
                        state['F10_qm'] = F10_qm
                    else:
                        state['r_t'] = 0.0
                        state['cos_z'] = 0.0
                        state['corr_u_D'] = 0.0
                        state['corr_qm_D'] = 0.0
                        state['F5_u'] = 0.0
                        state['F5_qm'] = 0.0
                        state['F10_u'] = 0.0
                        state['F10_qm'] = 0.0
                        
                p.grad.copy_(g_moq)
                moq_grads.append(g_moq)
                
                if 'p_before' not in state:
                    state['p_before'] = torch.zeros_like(p)
                state['p_before'].copy_(p)

        # Execute Untouched Official SOAP Step
        super().step()

        idx = 0
        for group in self.param_groups:
            for layer_idx, p in enumerate(group['params']):
                if p.grad is None:
                    continue
                state = self.state[p]
                
                raw_g = raw_grads[idx]
                g_moq = moq_grads[idx]
                idx += 1
                
                delta_theta = p - state['p_before']
                norm_update = float(torch.linalg.vector_norm(delta_theta))
                
                moq_diff_rel = float(torch.linalg.vector_norm(g_moq - raw_g)) / (float(torch.linalg.vector_norm(raw_g)) + 1e-12)
                
                diag = {
                    "layer": layer_idx,
                    "is_weight": p.dim() > 1, 
                    "norm_raw_G": float(torch.linalg.vector_norm(raw_g)),
                    "moq_diff_rel": moq_diff_rel,
                    "E_pred": state['current_moq_pred_error'],
                    "norm_update": norm_update,
                    
                    "r_t": state.get('r_t', 0.0),
                    "cos_z": state.get('cos_z', 0.0),
                    "corr_u_D": state.get('corr_u_D', 0.0),
                    "corr_qm_D": state.get('corr_qm_D', 0.0),
                    "F5_u": state.get('F5_u', 0.0),
                    "F5_qm": state.get('F5_qm', 0.0),
                    "F10_u": state.get('F10_u', 0.0),
                    "F10_qm": state.get('F10_qm', 0.0),
                }
                
                self.layer_diagnostics.append(diag)
                
        return loss

    def _get_orthogonal_matrix(self, mat_list):
        final = []
        for m in mat_list:
            if len(m) == 0:
                final.append([])
                continue
            m64 = m.to(torch.float64)
            m64 = 0.5 * (m64 + m64.T)
            _, Q = torch.linalg.eigh(m64 + 1e-30*torch.eye(m.shape[0], device=m.device))
            Q = torch.flip(Q, [1])
            final.append(Q.to(m.dtype))
        return final

    def _project(self, grad, Q_list):
        for mat in Q_list:
            if len(mat) > 0:
                grad = torch.tensordot(grad, mat, dims=[[0], [0]])
            else:
                permute_order = list(range(1, len(grad.shape))) + [0]
                grad = grad.permute(permute_order)
        return grad


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage14_audit(mu: float, K: int=1, steps: int=500, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule_long.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
        if len(schedule) < steps:
            schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
            torch.save(schedule, schedule_path)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = InstrumentedOfficialSOAPStage14(
        model.parameters(), 
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=mu, 
        damping=1e-3
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        err = relative_l2(model, device)
        
        step_data = {
            "step": step,
            "loss": float(loss.detach()),
            "rel_l2": float(err),
            "layers": opt.layer_diagnostics
        }
        history.append(step_data)
        
        if step % 25 == 0 or step == 1:
            print(f"Step {step:3d} | Loss {loss.item():.4e} | RelL2 {err:.4e}")

    return history


### Stage 14: Denominator Alignment Sweep

In [ ]:
for K in [20, 10, 5, 1]:
    print(f"\n{'='*40}\nRunning K={K} Denominator Alignment...\n{'='*40}")
    audit_moq = run_stage14_audit(mu=0.5, K=K, steps=500)
    with open(f"stage14_K{K}_soap_moq.json", "w") as f:
        json.dump(audit_moq, f, indent=2)
